# Module 1 : Uncertainty-Aware Priority Ranking

Instead of a single "risk score" per location, we predict a **priority band** (lower / median / upper) so operations teams see *how confident* the model is. A narrow band = confident; a wide band = needs more data.

- **Target** = a composite `priority_score` built from outcome signals (weighted incidents, injuries, weighted complaints, slow resolution).
- **Features** = location *context* only (road_type, road_condition, pedestrian_activity, near_school, near_hospital, zone) — so the model can rank even locations with sparse incident history.
- **Models** = three quantile Gradient Boosting regressors (5th / 50th / 95th percentile).

### 1. Load the CSV files

In [11]:
import pandas as pd
import numpy as np
import json

locations = pd.read_csv("../dataset/locations.csv")
incidents = pd.read_csv("../dataset/incidents.csv")
complaints = pd.read_csv("../dataset/public_complaints.csv")

# Clean IDs so joins are reliable
for df in [locations, incidents, complaints]:
    df["location_id"] = df["location_id"].astype(str).str.strip()

print("locations:", locations.shape)
print("incidents:", incidents.shape)
print("complaints:", complaints.shape)

locations: (1000, 10)
incidents: (10000, 9)
complaints: (8000, 7)


### 2. Aggregate to one row per location

Same weighting scheme as Module 3 so the two modules stay consistent.

In [12]:
severity_weights = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
priority_weights = {"Low": 1, "Normal": 2, "High": 3}

def apply_weights(series, mapping):
    values = series.astype("string").str.strip().str.lower()
    mapping = {k.lower(): v for k, v in mapping.items()}
    return values.map(mapping).fillna(1).astype(float)

incidents["incident_weight"] = apply_weights(incidents["severity"], severity_weights)
complaints["complaint_weight"] = apply_weights(complaints["priority_reported"], priority_weights)

incident_summary = (
    incidents.groupby("location_id")
    .agg(
        incident_count=("location_id", "size"),
        weighted_incidents=("incident_weight", "sum"),
        injury_count=("injuries", "sum"),
    )
    .reset_index()
)

complaint_summary = (
    complaints.groupby("location_id")
    .agg(
        complaint_count=("location_id", "size"),
        weighted_complaints=("complaint_weight", "sum"),
        avg_days_to_resolution=("days_to_resolution", "mean"),
    )
    .reset_index()
)

features = (
    locations
    .merge(incident_summary, on="location_id", how="left")
    .merge(complaint_summary, on="location_id", how="left")
)

count_columns = [
    "incident_count", "weighted_incidents", "injury_count",
    "complaint_count", "weighted_complaints",
]
features[count_columns] = features[count_columns].fillna(0)
features["avg_days_to_resolution"] = features["avg_days_to_resolution"].fillna(0)

print(features.shape)
features.head(3)

(1000, 16)


,location_id,zone,road_type,road_condition,population_density_per_km2,near_school,near_hospital,pedestrian_activity,latitude,longitude,incident_count,weighted_incidents,injury_count,complaint_count,weighted_complaints,avg_days_to_resolution
0,L0001,ZONE_19,Residential,Good,8597,0,0,Medium,16.516403,80.463890,11,23.0,5,10,23.0,8.540000
1,L0002,ZONE_14,Collector,Poor,1343,0,1,High,16.384392,80.429824,6,13.0,2,13,25.0,7.553846
2,L0003,ZONE_17,Local,Poor,14911,0,0,Medium,16.627553,80.440903,13,30.0,1,7,15.0,4.557143


### 3. Build the target `priority_score`

We z-score each outcome signal (so no single column dominates by scale) and sum them. This is what the model learns to predict.

In [13]:
def z_score(series):
    std = series.std(ddof=0)
    if pd.isna(std) or std == 0:
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std

target_signals = ["weighted_incidents", "injury_count", "weighted_complaints", "avg_days_to_resolution"]
for col in target_signals:
    features[col + "_z"] = z_score(features[col])

features["priority_score"] = features[[c + "_z" for c in target_signals]].sum(axis=1)

features[["location_id", "priority_score"]].describe()

,priority_score
count,1.000000e+03
mean,-6.501466e-16
std,2.241357e+00
min,-6.729223e+00
25%,-1.557806e+00
50%,-3.985023e-02
75%,1.435665e+00
max,9.746175e+00


### 4. Build the feature matrix (context only)

We deliberately **exclude** the outcome columns here — the model predicts priority from a location's physical/context attributes, giving it cold-start ability on sparse locations.

In [14]:
categorical = ["road_type", "road_condition", "pedestrian_activity", "zone"]
numeric = ["near_school", "near_hospital", "population_density_per_km2"]

# keep only numeric cols that actually exist in this dataset
numeric = [c for c in numeric if c in features.columns]

X = pd.get_dummies(features[categorical + numeric], columns=categorical)
y = features["priority_score"]

print("feature matrix:", X.shape)
X.head(3)

feature matrix: (1000, 35)


,near_school,near_hospital,population_density_per_km2,road_type_Arterial,road_type_Collector,road_type_Highway,road_type_Local,road_type_Residential,road_condition_Critical,road_condition_Fair,...,zone_ZONE_11,zone_ZONE_12,zone_ZONE_13,zone_ZONE_14,zone_ZONE_15,zone_ZONE_16,zone_ZONE_17,zone_ZONE_18,zone_ZONE_19,zone_ZONE_20
0,0,0,8597,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
1,0,1,1343,False,True,False,False,False,False,False,...,False,False,False,True,False,False,False,False,False,False
2,0,0,14911,False,False,False,True,False,False,False,...,False,False,False,False,False,False,True,False,False,False


### 5. Train three quantile models (lower / median / upper band)

In [15]:
from sklearn.ensemble import GradientBoostingRegressor

quantiles = {"band_low": 0.05, "predicted": 0.50, "band_high": 0.95}
models = {}

for name, alpha in quantiles.items():
    m = GradientBoostingRegressor(
        loss="quantile", alpha=alpha,
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0,
    )
    m.fit(X, y)
    models[name] = m
    features[name] = m.predict(X)

# enforce ordering (quantile models can occasionally cross)
features["band_low"] = features[["band_low", "predicted"]].min(axis=1)
features["band_high"] = features[["band_high", "predicted"]].max(axis=1)
features["band_width"] = features["band_high"] - features["band_low"]

features[["location_id", "band_low", "predicted", "band_high", "band_width"]].head(5)

,location_id,band_low,predicted,band_high,band_width
0,L0001,-3.749194,0.148474,3.700935,7.450128
1,L0002,-3.316152,-0.027864,3.721289,7.037441
2,L0003,-3.644817,-0.103839,3.984094,7.628912
3,L0004,-3.514399,-0.077302,3.992661,7.507060
4,L0005,-3.564745,0.165747,3.541788,7.106533


### 6. Validate — pinball loss (per quantile) + band coverage

There is no usable time axis at the per-location aggregation level, so we use K-fold CV. Band coverage should land near 90% (the 5–95 interval).

In [16]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_pinball_loss

kf = KFold(n_splits=5, shuffle=True, random_state=0)

for name, alpha in quantiles.items():
    losses = []
    for tr, te in kf.split(X):
        m = GradientBoostingRegressor(
            loss="quantile", alpha=alpha,
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0,
        )
        m.fit(X.iloc[tr], y.iloc[tr])
        preds = m.predict(X.iloc[te])
        losses.append(mean_pinball_loss(y.iloc[te], preds, alpha=alpha))
    print(f"{name:10s} (q={alpha}) avg pinball loss: {np.mean(losses):.4f}")

coverage = ((y >= features["band_low"]) & (y <= features["band_high"])).mean()
print(f"\nband coverage (target ~0.90): {coverage:.3f}")

band_low   (q=0.05) avg pinball loss: 0.2320
predicted  (q=0.5) avg pinball loss: 0.9015
band_high  (q=0.95) avg pinball loss: 0.2552

band coverage (target ~0.90): 0.898


### 7. Rank, add a plain-English confidence note, and export

Output schema mirrors `module3_divergence.json` so the frontend can consume both the same way.

In [17]:
features["priority_rank"] = features["predicted"].rank(ascending=False, method="first").astype(int)

# confidence label from band width terciles
q33, q66 = features["band_width"].quantile([0.33, 0.66])
def confidence_label(w):
    if w <= q33: return "high"
    if w <= q66: return "medium"
    return "low"
features["confidence"] = features["band_width"].apply(confidence_label)

def explain(row):
    return (
        f"Predicted priority {row['predicted']:.2f} "
        f"(range {row['band_low']:.2f} to {row['band_high']:.2f}, {row['confidence']} confidence). "
        f"Driven by road_condition={row['road_condition']}, "
        f"pedestrian_activity={row['pedestrian_activity']}, road_type={row['road_type']}."
    )
features["explanation"] = features.apply(explain, axis=1)

features = features.sort_values("priority_rank").reset_index(drop=True)

output_columns = [
    "location_id", "zone", "latitude", "longitude",
    "road_type", "road_condition", "near_school", "near_hospital", "pedestrian_activity",
    "incident_count", "weighted_incidents", "injury_count",
    "complaint_count", "weighted_complaints", "avg_days_to_resolution",
    "priority_score", "band_low", "predicted", "band_high", "band_width",
    "confidence", "explanation", "priority_rank",
]

map_output = features[output_columns].copy()
map_output = map_output.replace([np.inf, -np.inf], np.nan)

map_output.to_json("module1_priority.json", orient="records", indent=2)

print(f"Exported {len(map_output)} locations")
print("Saved: module1_priority.json")
print(map_output.head(3).to_json(orient="records", indent=2))

Exported 1000 locations
Saved: module1_priority.json
[
  {
    "location_id":"L0693",
    "zone":"ZONE_12",
    "latitude":16.540358,
    "longitude":80.442046,
    "road_type":"Residential",
    "road_condition":"Poor",
    "near_school":1,
    "near_hospital":1,
    "pedestrian_activity":"Medium",
    "incident_count":11,
    "weighted_incidents":22.0,
    "injury_count":2,
    "complaint_count":13,
    "weighted_complaints":29.0,
    "avg_days_to_resolution":6.3,
    "priority_score":2.6107963069,
    "band_low":-3.9793494225,
    "predicted":2.3258703103,
    "band_high":2.8984202871,
    "band_width":6.8777697095,
    "confidence":"high",
    "explanation":"Predicted priority 2.33 (range -3.98 to 2.90, high confidence). Driven by road_condition=Poor, pedestrian_activity=Medium, road_type=Residential.",
    "priority_rank":1
  },
  {
    "location_id":"L0943",
    "zone":"ZONE_18",
    "latitude":16.43259,
    "longitude":80.292599,
    "road_type":"Residential",
    "road_conditio